# Lab -- Dataset Curation with Multiple Annotators

Intended to accompany the lecture on Dataset Creation and Curation, this notebook contains exercises to analyze an existing classification dataset labeled by multiple annotators (e.g. collected via crowdsourcing).

You may find it helpful to first install the following dependencies:

In [1]:
!pip install cleanlab
# We originally used the version: cleanlab==2.2.0
# This automatically installs other required packages like numpy, pandas, sklearn

In [2]:
import numpy as np
import pandas as pd

## Analyzing dataset labeled by multiple annotators

We simulate a small classification dataset (3 classes, 2-dimensional features) with ground truth labels that are then hidden from our analysis. The analysis is conducted on labels from noisy annotators whose labels are derived from the ground truth labels, but with some probability of error in each annotated label where the probability is determined by the underlying quality of the annotator. In subsequent exercises, you should assume the ground truth labels and the true annotator qualities are unknown to you.

In [63]:
## You don't need to understand this cell, it's just used for generating the dataset

SEED = 123  # for reproducibility
np.random.seed(seed=SEED)

def make_data(sample_size = 300, num_annotators=50):
    """ Produce a 3-class classification dataset with 2-dimensional features and multiple noisy annotations per example. """
      # total number of data annotators
    class_frequencies = [0.5, 0.25, 0.25]
    sizes=[int(np.ceil(freq*sample_size)) for freq in class_frequencies]  # number of examples belonging to each class
    good_annotator_quality = 0.6
    bad_annotator_quality = 0.3
    
    # Underlying statistics of the datset (unknown to you)
    means=[[3, 2], [7, 7], [0, 8]]
    covs=[[[5, -1.5], [-1.5, 1]], [[1, 0.5], [0.5, 4]], [[5, 1], [1, 5]]]
    
    m = len(means)  # number of classes
    n = sum(sizes)
    local_data = []
    labels = []

    # Generate features and true labels
    for idx in range(m):
        local_data.append(
            np.random.multivariate_normal(mean=means[idx], cov=covs[idx], size=sizes[idx])
        )
        labels.append(np.array([idx for i in range(sizes[idx])]))
    X_train = np.vstack(local_data)
    true_labels_train = np.hstack(labels)

    # Generate noisy labels from each annotator
    s = pd.DataFrame(
        np.vstack(
            [
                generate_noisy_labels(true_labels_train, good_annotator_quality)
                if i < num_annotators - 10  # last 10 annotators are worse
                else generate_noisy_labels(true_labels_train, bad_annotator_quality)
                for i in range(num_annotators)
            ]
        ).transpose()
    )

    # Each annotator only labels approximately 10% of the dataset (unlabeled points represented with NaN)
    s = s.apply(lambda x: x.mask(np.random.random(n) < 0.9)).astype("Int64")
    s.dropna(axis=1, how="all", inplace=True)
    s.columns = ["A" + str(i).zfill(4) for i in range(1, num_annotators+1)]
    # Drop rows not annotated by anybody
    row_NA_check = pd.notna(s).any(axis=1)
    X_train = X_train[row_NA_check]
    true_labels_train = true_labels_train[row_NA_check]
    multiannotator_labels = s[row_NA_check].reset_index(drop=True)
    # Shuffle the rows of the dataset
    shuffled_indices = np.random.permutation(len(X_train))
    return {
        "X_train": X_train[shuffled_indices],
        "true_labels_train": true_labels_train[shuffled_indices],
        "multiannotator_labels": multiannotator_labels.iloc[shuffled_indices],
    }

def generate_noisy_labels(true_labels, annotator_quality):
    """ Randomly flips each true label to a different class with probability that depends on annotator_quality. """
    n = len(true_labels)
    m = np.max(true_labels) + 1  # number of classes
    annotated_labels = np.random.randint(low=0, high=3, size=n)
    correctly_labeled_indices = np.random.random(n) < annotator_quality
    annotated_labels[correctly_labeled_indices] = true_labels[correctly_labeled_indices]
    return annotated_labels

In [96]:
data_dict = make_data(sample_size = 300, num_annotators=50)

X = data_dict["X_train"]
multiannotator_labels = data_dict["multiannotator_labels"]
true_labels = data_dict["true_labels_train"] # used for comparing the accuracy of consensus labels

Let's view the first few rows of the data used for this exercise. Here are the labels selected by each annotator for the first few examples. Here each example is a row, and the annotators are columns. Not all annotators labeled each example; valid class annotations from those that did label the example are integers (either 0, 1, or 2 for our 3 classes), and otherwise the annotation is left as `NA` if a particular annotator did not label a particular example.

In [69]:
multiannotator_labels.head()

,A0001,A0002,A0003,A0004,A0005,A0006,A0007,A0008,A0009,A0010,...,A0491,A0492,A0493,A0494,A0495,A0496,A0497,A0498,A0499,A0500
170,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0
936,<NA>,<NA>,<NA>,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
113,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1009,<NA>,<NA>,0,<NA>,<NA>,2,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2,2
1149,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2,<NA>,<NA>


Here are the corresponding 2D data features for these examples:

In [10]:
X[:5]

array([[ 5.18907565,  1.66530173],
       [-0.14711999, 10.27517622],
       [ 5.11153752,  2.12769331],
       [ 6.00969072,  4.93630588],
       [ 4.47184928,  1.59260315]])

### Train model with cross-validation

In this exercise, we consider the simple K Nearest Neighbors classification model, which produces predicted class probabilities for a particular example via a (weighted) average of the labels of the K closest examples. We will train this model via cross-validation and use it to produce held-out predictions for each example in our dataset.

In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict

def train_model(labels_to_fit):
    """ Trains a simple feedforward neural network model on the data features X with y = labels_to_fit, via cross-validation.
        Returns out-of-sample predicted class probabilities for each example in the dataset
        (from a copy of model that was never trained on this example).
        Also evaluates the held-out class predictions against ground truth labels.
    """
    num_crossval_folds = 5  # number of folds of cross-validation
    # model = MLPClassifier(max_iter=1000, random_state=SEED)
    model = KNeighborsClassifier(weights="distance")
    pred_probs = cross_val_predict(
        estimator=model, X=X, y=labels_to_fit, cv=num_crossval_folds, method="predict_proba"
    )
    class_predictions = np.argmax(pred_probs, axis=1)
    held_out_accuracy = np.mean(class_predictions == true_labels)
    print(f"Accuracy of held-out model predictions against ground truth labels: {held_out_accuracy}")
    return pred_probs

Here we demonstrate how to train and evaluate this model. Note that the evaluation is against ground truth labels, which you wouldn't have in real applications, so this evaluation is just for demonstration purposes. We'll first fit this model using labels comprised of one randomly selected annotation for each example.

In [75]:
labels_from_random_annotators = true_labels.copy()
for i in range(len(multiannotator_labels)):
    annotations_for_example_i = multiannotator_labels.iloc[i][pd.notna(multiannotator_labels.iloc[i])]
    labels_from_random_annotators[i] = np.random.choice(annotations_for_example_i.values)

print(f"Accuracy of random annotators' labels against ground truth labels: {np.mean(labels_from_random_annotators == true_labels)}")
pred_probs_from_model_fit_to_random_annotators = train_model(labels_to_fit = labels_from_random_annotators)


Accuracy of random annotators' labels against ground truth labels: 0.7269230769230769
Accuracy of held-out model predictions against ground truth labels: 0.8430769230769231


We can also fit this model using the ground truth labels (which you would not be able to in practice), just to see how good it could be:

In [76]:
pred_probs_from_unrealistic_model_fit_to_true_labels = train_model(labels_to_fit = true_labels)

Accuracy of held-out model predictions against ground truth labels: 0.9792307692307692


## Exercise 1

Compute majority-vote consensus labels for each example from the data in `multiannotator_labels`. Think about how to best break ties!

- Evaluate the accuracy of these majority-vote consensus labels against the ground truth labels.
- Also set these as `labels_to_fit` in `train_model()` to see the resulting model's accuracy when trained with majority vote consensus labels.
- Estimate the quality of annotator (how accurate their labels tend to be overall) using only these majority-vote consensus labels (assume the ground truth labels are unavailable as they would be in practice). Who do you guess are the worst 10 annotators?

In [137]:
import tqdm
## Code your solution here
#np.nanargmax(np.array(multiannotator_labels), axis=0)
annotator_correct = {}
annotator_incorrect = {}

for _, row in tqdm.tqdm(list(multiannotator_labels.iterrows())):
    row = row.to_dict()
    actual_values = {annotator: value for annotator, value in row.items() if value is not None}
    #print(actual_values)
    values = [val for _, val in actual_values.items()]
    
    # Calculate the number of times each label appears
    # numpy magic
    label_counts_dict = {int(value): int(count) for value, count in zip(*np.unique(values, return_counts=True))}
    label_counts = list(sorted(label_counts_dict.items(), key=lambda p: -p[1]))
    
    # Ties
    if len(label_counts) >= 2 and label_counts[0][1] == label_counts[1][1]:
        top_count = label_counts[0][1]
        #print(actual_values)
        top_guesses = [label for label, count in label_counts if count == top_count]
        #print("TIE,", top_count, "for", len(top_guesses), "classes,", top_guesses)
        #print()
    else:
        majority_label = label_counts[0][0]
        #print(actual_values)
        #print("MAJORITY LABEL:", majority_label)
        agreement = label_counts[0][1] / len(actual_values)
        #print("agreement:", agreement)
        
        # Count reliability
        if len(actual_values) >= 2:
            for annotator, label in actual_values.items():
                if label == majority_label:
                    annotator_correct[annotator] = annotator_correct.get(annotator, 0.0) + 1
                else:
                    annotator_incorrect[annotator] = annotator_incorrect.get(annotator, 0.0) + 1
                    

alpha = 0.1
annotator_quality = dict()
for annotator in annotator_correct:
    corr, incorr = annotator_correct.get(annotator, 0.0), annotator_incorrect.get(annotator, 0.0)
    # Do a little smoothing
    corr, incorr = corr + alpha, incorr + alpha
    quality = corr / (corr + incorr)
    annotator_quality[annotator] = quality

labels_tiebreak = {}
annotator_quality = { ann: annotator_quality[ann] for ann in sorted(annotator_quality)}
annotator_quality

100%|██████████| 299/299 [00:00<00:00, 18189.29it/s]


{'A0001': 0.6655629139072847,
 'A0002': 0.7022058823529411,
 'A0003': 0.5,
 'A0004': 0.6242236024844721,
 'A0005': 0.6986754966887417,
 'A0006': 0.8546099290780141,
 'A0007': 0.7892561983471074,
 'A0008': 0.8044871794871794,
 'A0009': 0.7724358974358974,
 'A0010': 0.7702702702702703,
 'A0011': 0.9658385093167702,
 'A0012': 0.7923976608187134,
 'A0013': 0.7980132450331126,
 'A0014': 0.8373015873015872,
 'A0015': 0.7317880794701986,
 'A0016': 0.6939655172413793,
 'A0017': 0.7757352941176471,
 'A0018': 0.6242236024844721,
 'A0019': 0.81055900621118,
 'A0020': 0.6773049645390071,
 'A0021': 0.8191489361702128,
 'A0022': 0.7122641509433962,
 'A0023': 0.7757352941176471,
 'A0024': 0.8365384615384615,
 'A0025': 0.6939655172413793,
 'A0026': 0.7403846153846153,
 'A0027': 0.728494623655914,
 'A0028': 0.7560240963855421,
 'A0029': 0.8603603603603603,
 'A0030': 0.8305785123966942,
 'A0031': 0.7579365079365079,
 'A0032': 0.7370689655172413,
 'A0033': 0.9198473282442747,
 'A0034': 0.6652892561983471

In [153]:
# DO THE SAME THING AGAIN

# BUT BREAK TIES WRT ANNOTATOR QUALITY
import tqdm, random

ties_broken = 0
# tiebreak_strategy = "random"
tiebreak_strategy = "random"
assert tiebreak_strategy in ["annotator_quality", "random"]
labels = []
for _, row in tqdm.tqdm(list(multiannotator_labels.iterrows())):
    row = row.to_dict()
    actual_values = {annotator: value for annotator, value in row.items() if value is not None}
    #print(actual_values)
    values = [val for _, val in actual_values.items()]
    
    # Calculate the number of times each label appears
    # numpy magic
    label_counts_dict = {int(value): int(count) for value, count in zip(*np.unique(values, return_counts=True))}
    label_counts = list(sorted(label_counts_dict.items(), key=lambda p: -p[1]))
    
    # Ties
    if len(label_counts) >= 2 and label_counts[0][1] == label_counts[1][1]:
        epsilon = 0.00001
        top_count = label_counts[0][1]
        #print(actual_values)
        print("Tie;", actual_values)
        if tiebreak_strategy == "annotator_quality":
            for annotator in actual_values:
                quality = annotator_quality[annotator]
                label = actual_values[annotator]
                print(annotator, label, quality)
                very_small_tiebreak = np.random.rand() * epsilon * epsilon
                #print(very_small_tiebreak)
                label_counts_dict[label] = label_counts_dict[label] + epsilon * quality + very_small_tiebreak
            label_counts = list(sorted(label_counts_dict.items(), key=lambda p: -p[1]))    
            assert label_counts[0][1] != label_counts[1][1], f"label counts {label_counts}"

            best_label = label_counts[0][0]
            print("best label:", best_label)
            labels.append(best_label)
        else:
            top_guesses = [label for label, count in label_counts if count == top_count]
            best_label = np.random.choice(top_guesses)
            labels.append(best_label)
        ties_broken += 1
    else:
        majority_label = label_counts[0][0]
        labels.append(majority_label)
       
labels_tiebreak[tiebreak_strategy] = labels 
print()
print("In the data,", ties_broken, "ties were broken via", tiebreak_strategy)
#labels

100%|██████████| 299/299 [00:00<00:00, 15884.90it/s]

Tie; {'A0026': 2, 'A0043': 0}
Tie; {'A0010': 2, 'A0015': 0, 'A0023': 1}
Tie; {'A0025': 0, 'A0033': 2}
Tie; {'A0001': 0, 'A0007': 0, 'A0045': 2, 'A0046': 2}
Tie; {'A0002': 0, 'A0012': 1, 'A0021': 0, 'A0041': 2, 'A0045': 2}
Tie; {'A0010': 1, 'A0021': 0, 'A0032': 2}
Tie; {'A0008': 2, 'A0039': 0}
Tie; {'A0003': 0, 'A0026': 2}
Tie; {'A0009': 2, 'A0015': 1, 'A0019': 0, 'A0024': 2, 'A0027': 0}
Tie; {'A0037': 1, 'A0039': 0}
Tie; {'A0027': 0, 'A0028': 0, 'A0031': 1, 'A0033': 1}
Tie; {'A0005': 2, 'A0014': 0, 'A0015': 1, 'A0026': 2, 'A0041': 0}
Tie; {'A0017': 1, 'A0029': 0, 'A0038': 1, 'A0047': 0}
Tie; {'A0003': 2, 'A0009': 0, 'A0017': 0, 'A0032': 1, 'A0033': 1, 'A0035': 2, 'A0041': 0, 'A0049': 1}
Tie; {'A0023': 2, 'A0046': 0}
Tie; {'A0009': 2, 'A0014': 0}
Tie; {'A0002': 1, 'A0004': 1, 'A0017': 2, 'A0020': 2, 'A0023': 0, 'A0025': 0, 'A0028': 0, 'A0032': 2}
Tie; {'A0013': 2, 'A0018': 0}
Tie; {'A0012': 0, 'A0030': 2, 'A0045': 2, 'A0050': 0}
Tie; {'A0004': 2, 'A0009': 1, 'A0024': 2, 'A0027': 0, 'A00

In [154]:
print(f"Accuracy of random tiebreak labels against ground truth labels: {np.mean(labels_tiebreak["random"] == true_labels)}")
print()

print(f"Accuracy of annotator_quality tiebreak labels against ground truth labels: {np.mean(labels_tiebreak["annotator_quality"] == true_labels)}")
print()

print("Labels with Random tiebreaking")
pred_probs_from_model_fit_to_random_annotators = train_model(labels_to_fit = labels_tiebreak["random"])
print()
print("Labels with Annotator quality tiebreaking")
pred_probs_from_model_fit_to_random_annotators = train_model(labels_to_fit = labels_tiebreak["annotator_quality"])
print()
print("Ground truth labels")
pred_probs_from_unrealistic_model_fit_to_true_labels = train_model(labels_to_fit = true_labels)

Accuracy of random tiebreak labels against ground truth labels: 0.882943143812709

Accuracy of annotator_quality tiebreak labels against ground truth labels: 0.862876254180602

Labels with Random tiebreaking
Accuracy of held-out model predictions against ground truth labels: 0.9565217391304348

Labels with Annotator quality tiebreaking
Accuracy of held-out model predictions against ground truth labels: 0.9331103678929766

Ground truth labels
Accuracy of held-out model predictions against ground truth labels: 0.9732441471571907


### Result summary of exercise 1

- Both of the tie breaking methods improve the hold-out performance over the unprocessed noisy labels.
- The improvement is quite substantial: accuracy rises from 0.843 -> 0.93 and 0.94
- As to be expected, the performance of the random tiebreaking method fluctuates a little depending on the randomization (0.93-0.95). 
- Training on the ground truth labels yields a marginally better performance: 0.93-0.94 vs. 0.97

## Exercise 2

Estimate consensus labels for each example from the data in `multiannotator_labels`, this time using the CROWDLAB algorithm. You may find it helpful to reference: https://docs.cleanlab.ai/stable/tutorials/multiannotator.html
Recall that CROWDLAB requires out of sample predicted class probabilities from a trained classifier. You may use the `pred_probs` from your model trained on majority-vote consensus labels or our randomly-selected annotator labels. Which do you think would be better to use?

- Evaluate the accuracy of these CROWDLAB consensus labels against the ground truth labels.
- Also set these as `labels_to_fit` in `train_model()` to see the resulting model's accuracy when trained with CROWDLAB consensus labels.
- Estimate the quality of annotator (how accurate their labels tend to be overall) using CROWDLAB (assume the ground truth labels are unavailable as they would be in practice). Who do you guess are the worst 10 annotators based on this method?

In [160]:
## Code your solution here
from cleanlab.multiannotator import get_majority_vote_label
crowdlab_labels = get_majority_vote_label(multiannotator_labels, pred_probs_from_model_fit_to_random_annotators)

print(f"Accuracy of random tiebreak labels against ground truth labels: {np.mean(labels_tiebreak["random"] == true_labels)}")
print()

print(f"Accuracy of annotator_quality tiebreak labels against ground truth labels: {np.mean(labels_tiebreak["annotator_quality"] == true_labels)}")
print()

print(f"Accuracy of CROWDLAB label estimation against ground truth labels: {np.mean(crowdlab_labels== true_labels)}")
print()

Accuracy of random tiebreak labels against ground truth labels: 0.882943143812709

Accuracy of annotator_quality tiebreak labels against ground truth labels: 0.862876254180602

Accuracy of CROWDLAB label estimation against ground truth labels: 0.9163879598662207



In [161]:
print("Hold-out performance on train labels with CROWDLAB label estimation")
pred_probs_from_model_fit_to_random_annotators = train_model(labels_to_fit = crowdlab_labels)
print()
print("Ground truth labels")
pred_probs_from_unrealistic_model_fit_to_true_labels = train_model(labels_to_fit = true_labels)

Hold-out performance on train labels with CROWDLAB label estimation
Accuracy of held-out model predictions against ground truth labels: 0.9732441471571907

Ground truth labels
Accuracy of held-out model predictions against ground truth labels: 0.9732441471571907


### Result summary of exercise 2

- The accuracy of the CROWDLAB label estimation was higher than annotator_quality and random tiebreaking, but only somewhat
- The inferred labels work very well in terms of training, the performance was the same as with ground truth labels